# Static anomaly screening in ICU stay records

This notebook covers the record-level part of the accepted project. It turns repeated vital-sign readings into one row per ICU stay and compares Isolation Forest with a small autoencoder. The scores help prioritize records for review; they do not determine whether a record is wrong.

The eICU demo has no verified data-error labels. The controlled evaluation below adds known synthetic corruptions to held-out stays. It checks only those artificial changes and should not be described as real-world accuracy.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
from sklearn.neural_network import MLPRegressor

## 2. Load the vital-sign table

In [ ]:
# Download the eICU demo and put vitalPeriodic.csv.gz in data/eicu/.
# Update this path if your files are stored elsewhere.
DATA_PATH = Path("../data/eicu/vitalPeriodic.csv.gz")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/eicu/vitalPeriodic.csv.gz")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Place vitalPeriodic.csv.gz in data/eicu/ (see README.md for the source and setup)."
    )

raw = pd.read_csv(DATA_PATH, compression="gzip", low_memory=False)
print("Rows and columns:", raw.shape)
display(raw.head())
print("Columns:", raw.columns.tolist())

## 3. Explore data quality and select features

In [ ]:
# The expected eICU table includes repeated observations per ICU stay.
# We retain only columns present in this version of the file.
candidate_features = [
    "temperature", "sao2", "heartrate", "respiration",
    "systemicsystolic", "systemicdiastolic", "systemicmean", "glucose"
]
id_col = "patientunitstayid"
if id_col not in raw.columns:
    raise ValueError(f"Expected stay key {id_col!r} was not found; inspect the dataset schema.")
features = [c for c in candidate_features if c in raw.columns]
if len(features) < 3:
    raise ValueError(f"Too few expected vital-sign columns were found: {features}")

print("Selected vital-sign columns:", features)
display(raw[features].describe().T)
missing = raw[features].isna().mean().sort_values(ascending=False)
display(missing.rename("missing_fraction").to_frame())
plt.figure(figsize=(9, 3))
missing.plot(kind="bar")
plt.ylabel("Fraction missing")
plt.title("Missingness in selected vital-sign measurements")
plt.tight_layout()
plt.show()

## 4. Aggregate observations into one row per ICU stay

The unit of analysis is the ICU stay. We summarize each vital sign with its median, minimum, maximum, and measurement count. Identifiers are retained only to join results back for review; they are excluded from model features. Measurements may differ by unit or context, so inspect the data dictionary and output ranges before interpreting a flag.

In [ ]:
work = raw[[id_col] + features].copy()
for col in features:
    work[col] = pd.to_numeric(work[col], errors="coerce")

agg = work.groupby(id_col)[features].agg(["median", "min", "max", "count"])
agg.columns = [f"{feature}_{stat}" for feature, stat in agg.columns]
stay_features = agg.reset_index()
model_cols = [c for c in stay_features.columns if c != id_col]
print("Aggregated stay-level shape:", stay_features.shape)
display(stay_features.head())
display(stay_features[model_cols].describe().T)

## 5. Prepare numeric features

In [ ]:
X_raw = stay_features[model_cols].replace([np.inf, -np.inf], np.nan)
# Remove entirely empty features, then impute remaining missing values with medians.
usable_cols = X_raw.columns[X_raw.notna().any(axis=0)].tolist()
X_raw = X_raw[usable_cols]
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X_raw), columns=usable_cols)
scaler = RobustScaler()
X = pd.DataFrame(scaler.fit_transform(X_imputed), columns=usable_cols)
print("Model matrix:", X.shape)
print("Missing after imputation:", int(X.isna().sum().sum()))

## 6. Fit Isolation Forest and an autoencoder

Isolation Forest isolates records that are unusual in the feature space. The autoencoder learns to reconstruct the features; a larger reconstruction error can indicate a stay that differs from patterns seen during training. Both methods are unsupervised. The 5% flag rate is a comparison setting for this exercise, not a threshold validated for health data review.

In [ ]:
# Robust univariate baseline: maximum absolute median/MAD score.
med = X_imputed.median(axis=0)
mad = (X_imputed - med).abs().median(axis=0).replace(0, np.nan)
robust_z = ((X_imputed - med) / (1.4826 * mad)).replace([np.inf, -np.inf], np.nan).fillna(0)
baseline_score = robust_z.abs().max(axis=1).to_numpy()

contamination = 0.05
if_model = IsolationForest(
    n_estimators=200, contamination=contamination,
    random_state=RANDOM_STATE, n_jobs=-1
)
if_model.fit(X)
if_score = -if_model.decision_function(X)

# A narrow hidden layer compresses and reconstructs the stay features.
hidden_width = max(2, min(8, X.shape[1] // 2))
autoencoder = MLPRegressor(hidden_layer_sizes=(hidden_width,), activation="relu",
                           solver="adam", max_iter=500, early_stopping=True,
                           random_state=RANDOM_STATE)
autoencoder.fit(X, X)
ae_score = np.mean((X.to_numpy() - autoencoder.predict(X)) ** 2, axis=1)

results = stay_features[[id_col]].copy()
results["baseline_score"] = baseline_score
results["iforest_score"] = if_score
results["autoencoder_score"] = ae_score
results["baseline_flag"] = results.baseline_score >= results.baseline_score.quantile(1-contamination)
results["iforest_flag"] = if_model.predict(X) == -1
results["autoencoder_flag"] = results.autoencoder_score >= results.autoencoder_score.quantile(1-contamination)
print("Flag counts:")
display(results[["baseline_flag", "iforest_flag"]].sum().rename("flagged_stays").to_frame())
print("Overlap:", int((results.baseline_flag & results.iforest_flag).sum()))
display(results.sort_values("iforest_score", ascending=False).head(10))

## 7. Controlled evaluation with synthetic corruptions

Because there are no verified anomalous-record labels, this experiment creates a test set from held-out real stays and injects large deviations into one randomly chosen feature per copy. The clean reference rows receive label 0 and their corrupted copies label 1. This assesses only these injected corruption patterns. Split before fitting preprocessing and models to avoid evaluating on training records.

In [ ]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw = train_test_split(
    X_raw, test_size=0.30, random_state=RANDOM_STATE
)
# Fit imputation/scaling on training rows only for this evaluation.
eval_imputer = SimpleImputer(strategy="median")
train_imp = eval_imputer.fit_transform(X_train_raw)
test_imp = eval_imputer.transform(X_test_raw)
eval_scaler = RobustScaler().fit(train_imp)
train_scaled = eval_scaler.transform(train_imp)
test_scaled = eval_scaler.transform(test_imp)

eval_if = IsolationForest(n_estimators=200, contamination=contamination,
                          random_state=RANDOM_STATE, n_jobs=-1)
eval_if.fit(train_scaled)
eval_ae = MLPRegressor(hidden_layer_sizes=(max(2, min(8, train_scaled.shape[1] // 2)),),
                       activation="relu", solver="adam", max_iter=500,
                       early_stopping=True, random_state=RANDOM_STATE)
eval_ae.fit(train_scaled, train_scaled)

test_df = pd.DataFrame(test_imp, columns=usable_cols)
train_df = pd.DataFrame(train_imp, columns=usable_cols)
med_train = train_df.median()
mad_train = (pd.DataFrame(train_imp, columns=usable_cols) - med_train).abs().median().replace(0, np.nan)

clean = test_df.copy()
corrupt = test_df.copy()
rng = np.random.default_rng(RANDOM_STATE)
row_ids = rng.permutation(len(corrupt))
col_ids = rng.integers(0, corrupt.shape[1], size=len(corrupt))
for row, col in zip(row_ids, col_ids):
    col_name = corrupt.columns[col]
    scale = float(mad_train[col_name])
    if not np.isfinite(scale) or scale == 0:
        scale = float(pd.Series(train_imp[:, col]).std() or 1.0)
    # Push one feature far from its training median; alternate directions.
    direction = -1 if rng.random() < 0.5 else 1
    corrupt.iat[row, col] = med_train[col_name] + direction * 8 * scale

eval_X = pd.concat([clean, corrupt], ignore_index=True)
y_eval = np.r_[np.zeros(len(clean), dtype=int), np.ones(len(corrupt), dtype=int)]
eval_scaled = eval_scaler.transform(eval_X)
eval_if_scores = -eval_if.decision_function(eval_scaled)
eval_ae_scores = np.mean((eval_scaled - eval_ae.predict(eval_scaled)) ** 2, axis=1)
eval_z = ((eval_X - med_train) / (1.4826 * mad_train)).replace([np.inf, -np.inf], np.nan).fillna(0)
eval_baseline_scores = eval_z.abs().max(axis=1).to_numpy()

def summarize_scores(name, scores):
    return {"method": name,
            "roc_auc": roc_auc_score(y_eval, scores),
            "average_precision": average_precision_score(y_eval, scores)}

eval_summary = pd.DataFrame([
    summarize_scores("MAD baseline", eval_baseline_scores),
    summarize_scores("Isolation Forest", eval_if_scores),
    summarize_scores("Autoencoder", eval_ae_scores),
])
display(eval_summary)
print("Evaluation pairs:", len(clean), "clean +", len(corrupt), "synthetically corrupted")

## 8. Inspect flagged stays and interpret cautiously

In [ ]:
flagged_ids = results.loc[results.iforest_flag, id_col]
flagged = stay_features[stay_features[id_col].isin(flagged_ids)].copy()
flagged = flagged.merge(results[[id_col, "baseline_score", "iforest_score"]], on=id_col)
display(flagged.sort_values("iforest_score", ascending=False).head(15))

# Show the feature summaries most different from the overall median in the top flags.
top = flagged.sort_values("iforest_score", ascending=False).head(10)
summary_cols = [c for c in model_cols if c.endswith("_median")]
if len(top) and summary_cols:
    display(top[[id_col] + summary_cols].T)

## 9. Conclusion prompts

Complete this section after running the notebook:

- Which of the MAD baseline, Isolation Forest, and autoencoder ranked synthetic corruptions better, based on ROC-AUC and average precision?
- What does that tell you about the corruption patterns used here, and what does it **not** tell you about real data errors?
- How many stays did each method flag, and where did the methods agree?
- Which features were associated with the highest scores? Are they plausibly data quality issues, valid clinical extremes, or artifacts of missingness/aggregation?
- What additional labeled data or domain review would be needed before deployment?

**Do not describe the scores as clinical diagnoses or as validated detection accuracy.**

Compare these static results with the separate time-series analysis. Keep findings from the two datasets distinct.

## References

Johnson, A., Pollard, T., Badawi, O., & Raffa, J. (2019). *eICU Collaborative Research Database Demo (version 2.0).* PhysioNet. https://doi.org/10.13026/gxmm-es70

Dataset page: https://physionet.org/content/eicu-crd-demo/2.0/